In [3]:
import json
import random
from pathlib import Path


### Film Sentiment

In [4]:

MOVIES_DIR = Path("./movies")
DOCS_DIR = MOVIES_DIR / "docs"
TRAIN_PATH = MOVIES_DIR / "train.jsonl"


N = 3
SEED = 42

def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def normalize_label(y: str) -> str:
    y = str(y).strip().lower()
    if y in {"pos", "positive"}:
        return "positive"
    if y in {"neg", "negative"}:
        return "negative"
    raise ValueError(f"Unrecognized label: {y}")

def load_doc(docid: str) -> str:
    # docid in your copy can be like "negR_000.txt"
    p = DOCS_DIR / docid
    if p.suffix != ".txt":
        p = p.with_suffix(".txt")
    return p.read_text(encoding="utf-8", errors="replace").strip()

def make_prompt(text: str, label: str) -> str:
    # Standard, low-friction sentiment prompting
    # Ends with the corresponding label (as you requested)
    return (
        "Task: Determine whether the following movie review is positive or negative.\n\n"
        f"Review:\n{text}\n\n"
        "Sentiment: " + label
    )

# Load training annotations
train = list(read_jsonl(TRAIN_PATH))
assert len(train) >= N, f"Train set has only {len(train)} examples, need {N}"

# Sample without replacement
random.seed(SEED)
sampled = random.sample(train, N)

# Build prompts
prompts = []
for ex in sampled:
    docid = ex["annotation_id"]              # e.g., "negR_000.txt"
    text = load_doc(docid)
    label = normalize_label(ex["classification"])
    prompt = make_prompt(text, label)
    prompts.append({
        "annotation_id": docid,
        "label": label,
        "prompt": prompt
    })

# Print results
for i, item in enumerate(prompts, 1):
    print("=" * 80)
    # print(f"[{i}] {item['annotation_id']}  label={item['label']}")
    print(item["prompt"])
    print()


Task: Determine whether the following movie review is positive or negative.

Review:
as i write the review for the new hanks / ryan romantic comedy you 've got mail , i am acutely aware that i am typing it on a computer and sending it a billion miles away on the internet .
i am also aware that i have just spent the last 2 hours watching the world 's biggest paid commercial for america online .
and i wonder : is that so bad ?
well , the commercial part is .
as for the movie , well , as long as i can watch tom hanks and meg ryan , i think i 'll be okay .
to paraphrase james berardinelli , whose reviews i admire very much , tom hanks and meg ryan can act .
they are both wonderful , but for all of hanks ' glorious work in serious films , such as his magnificent performance in saving private ryan , and his glorious triumph in philadelphia , i like him best when he 's suitably obnoxious .
tom hanks is wonderful
when he is obnoxious in a romantic comedy when he 's going to get the girl : the 

### ESNLI: entail, neutral, contradiction


In [18]:
from datasets import load_dataset

ds = load_dataset("esnli/esnli", revision="refs/convert/parquet")
print(ds)

N = 500
SEED = 42

# e-SNLI label mapping: 0=entailment, 1=neutral, 2=contradiction
LABEL_NAMES = {0: "entail", 1: "neutral", 2: "contradiction"}

def make_nli_prompt(premise: str, hypothesis: str, label: str) -> str:
    """Create a prompt that elicits the correct NLI label from the LLM using label names directly."""
    return (
        "Task: Determine the relationship between the premise and hypothesis. "
        "Answer with exactly one word: entailment, neutral, or contradiction.\n\n"
        f"Premise: {premise} "
        f"Hypothesis: {hypothesis} "
        "Options:\n"
        "- entailment\n"
        "- neutral\n"
        "- contradiction\n\n"
        f"Answer: {label}"
    )

# Sample N examples without replacement from the training set
train_data = ds["train"]
assert len(train_data) >= N, f"Train set has only {len(train_data)} examples, need {N}"

random.seed(SEED)
indices = random.sample(range(len(train_data)), N)

# Build prompts (premise, hypothesis, label)
esnli_prompts = []
for idx in indices:
    ex = train_data[int(idx)]
    premise = ex["premise"]
    hypothesis = ex["hypothesis"]
    label = LABEL_NAMES[ex["label"]]
    prompt = make_nli_prompt(premise, hypothesis, label)
    esnli_prompts.append({
        "premise": premise,
        "hypothesis": hypothesis,
        "label": label,
        "answer": label,
        "prompt": prompt
    })

# Print results
for i, item in enumerate(esnli_prompts, 1):
    print("=" * 80)
    print(item["prompt"])
    print()



DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'explanation_1', 'explanation_2', 'explanation_3'],
        num_rows: 549367
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label', 'explanation_1', 'explanation_2', 'explanation_3'],
        num_rows: 9842
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label', 'explanation_1', 'explanation_2', 'explanation_3'],
        num_rows: 9824
    })
})
Task: Determine the relationship between the premise and hypothesis. Answer with exactly one word: entailment, neutral, or contradiction.

Premise: Some people in blue shirts are standing up at an even with big letters spelling out the word "KRUNCH". Hypothesis: The people are wearing blue. Options:
- entailment
- neutral
- contradiction

Answer: entail

Task: Determine the relationship between the premise and hypothesis. Answer with exactly one word: entailment, neutral, or contradiction.

Premise: A ma

In [19]:
# Save prompts as a list of dicts with singleton "prompt" item
prompt_only = [{"prompt": item["prompt"]} for item in esnli_prompts]
esnli_out_path = Path("esnli_prompts.json")
esnli_out_path.parent.mkdir(parents=True, exist_ok=True)
with open(esnli_out_path, "w", encoding="utf-8") as f:
    json.dump(prompt_only, f, indent=2, ensure_ascii=False)
print(f"Saved {len(prompt_only)} prompts to {esnli_out_path}")


Saved 500 prompts to esnli_prompts.json


### ESNLI: binary support (yes/no, exclude neutral)

In [38]:
def load_esnli_binary_support(n=500, seed=42, split="train"):
    """
    Load ESNLI and prompt LLM to answer whether premise supports hypothesis (yes/no).
    Excludes neutral (ambiguous) cases; only entailment -> yes, contradiction -> no.
    """
    from datasets import load_dataset
    ds = load_dataset("esnli/esnli", revision="refs/convert/parquet")
    data = ds[split]

    # Filter out neutral (label=1); keep entailment (0) and contradiction (2)
    valid_indices = [i for i in range(len(data)) if data[i]["label"] != 1]
    assert len(valid_indices) >= n, f"Need {n} examples, got {len(valid_indices)} non-neutral"

    random.seed(seed)
    sampled_idx = random.sample(valid_indices, n)

    def make_support_prompt(premise: str, hypothesis: str, label: str) -> str:
        return (
            f"Does the premise support the hypothesis? Answer yes or no."
            f"Premise: {premise}"
            f"Hypothesis: {hypothesis}"
            f"Answer: {label}"
        )

    # 0=entailment -> yes, 2=contradiction -> no
    label_map = {0: "yes", 2: "no"}

    prompts = []
    for i in sampled_idx:
        ex = data[int(i)]
        premise = ex["premise"]
        hypothesis = ex["hypothesis"]
        label = label_map[ex["label"]]
        prompt = make_support_prompt(premise, hypothesis, label)
        prompts.append({
            "premise": premise,
            "hypothesis": hypothesis,
            "label": label,
            "prompt": prompt
        })
    return prompts

# Example usage: load 500, save to JSON
esnli_binary = load_esnli_binary_support(n=500, seed=42)
out_path = Path("esnli_binary_support_prompts.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([{"prompt": p["prompt"]} for p in esnli_binary], f, indent=2, ensure_ascii=False)
print(f"Saved {len(esnli_binary)} prompts to {out_path}")

Saved 500 prompts to esnli_binary_support_prompts.json


### Translation dataset

In [22]:
from datasets import load_dataset

ds = load_dataset("opus_books", "de-en")  

In [23]:
def strip_trailing_punct(text):
    """Remove trailing punctuation from text."""
    return text.rstrip('.,;:?!\'")\u201d\u201c\u2019\u2018')

def build_prompt(de, en):
    """Build single-line prompt: Deutsch ... English ... (no line break)."""
    en_clean = strip_trailing_punct(en.strip())
    return f"Deutsch: {de} English: {en_clean}"

In [ ]:
# Build prompts: Deutsch ... English ... (single line, no break). English >= 15 words, strip trailing punct.
N = 500
SEED = 42
MIN_EN_WORDS = 15
train_data = ds["train"]
random.seed(SEED)
indices = list(range(len(train_data)))
random.shuffle(indices)
opus_prompts = []
for i in indices:
    if len(opus_prompts) >= N:
        break
    ex = train_data[int(i)]
    de, en = ex["translation"]["de"], ex["translation"]["en"]
    if len(en.split()) < MIN_EN_WORDS:
        continue
    prompt = build_prompt(de, en)
    opus_prompts.append({"prompt": prompt, "id": ex.get("id", str(i))})
assert len(opus_prompts) == N, f"Could not find {N} examples with English >= {MIN_EN_WORDS} words"

# Save to opus_de_en_prompts.json
out_path = Path("opus_de_en_prompts.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
prompt_only = [{"prompt": item["prompt"]} for item in opus_prompts]
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(prompt_only, f, indent=2, ensure_ascii=False)
print(f"Saved {len(prompt_only)} prompts to {out_path}")

Saved 500 prompts to opus_de_en_prompts.json


### LAMBADA

In [2]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/EleutherAI/lambada_openai/resolve/main/data/lambada_test_en.jsonl"
)

print(ds)


/home/jl3499/conda/LLM1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 5153
    })
})


In [6]:


N = 350
SEED = 42
MIN_WORDS = 50
QUOTE_CHARS = '"\'"\u201c\u201d\u2018\u2019'

def strip_trailing_punct(text):
    return text.rstrip('.,;:?!\'")\u201d\u201c\u2019\u2018')

def is_valid_example(text):
    if any(c in text for c in QUOTE_CHARS):
        return False
    if len(text.split()) < MIN_WORDS:
        return False
    return True

random.seed(SEED)
train_data = ds["train"]
indices = list(range(len(train_data)))
random.shuffle(indices)
samples = []
for i in indices:
    if len(samples) >= N:
        break
    ex = train_data[int(i)]
    text = ex["text"]
    print(len(text.split()))
    if not is_valid_example(text):
        continue
    cleaned = strip_trailing_punct(text.strip())
    samples.append({"text": cleaned})
assert len(samples) == N, f"Could not find {N} examples without quotes and with >= {MIN_WORDS} words"



49
49
80
88
71
67
58
54
71
53
43
52
54
52
57
112
50
57
45
63
50
52
64
64
63
42
84
44
70
49
56
59
53
77
54
67
80
50
71
57
51
50
56
56
64
50
55
45
62
73
53
55
56
71
58
69
60
55
52
89
63
59
69
62
73
42
63
49
62
72
69
65
50
52
65
61
52
48
74
49
44
53
60
46
50
51
69
54
64
63
46
53
47
59
70
61
55
49
59
54
79
45
52
72
65
53
73
43
70
42
50
63
64
55
52
46
66
51
58
41
56
65
62
41
55
39
76
60
66
82
51
51
57
74
68
53
59
38
56
46
61
70
76
58
63
61
48
58
68
56
46
65
63
54
59
62
65
42
50
65
40
66
71
68
61
48
46
47
60
50
74
70
48
59
53
59
50
52
51
63
80
46
84
50
43
52
56
43
68
70
62
69
54
59
46
61
51
56
48
72
52
60
59
39
50
56
62
55
46
80
53
50
58
45
63
62
76
45
53
64
64
56
53
82
65
51
46
63
52
61
40
47
58
47
57
67
52
59
63
79
59
53
55
51
46
52
60
52
73
62
58
59
55
58
70
47
50
59
60
67
54
66
56
64
60
56
59
81
54
69
51
60
90
43
58
62
56
50
57
54
74
63
48
54
70
80
98
58
91
75
43
59
56
51
52
80
72
55
63
76
61
59
76
70
45
46
48
63
54
63
65
62
51
62
54
56
31
49
61
58
54
49
61
69
47
45
52
50
64
45
41
47
48


In [ ]:
# out_path = Path("lambada_prompts.json")
# out_path.parent.mkdir(parents=True, exist_ok=True)
# with open(out_path, "w", encoding="utf-8") as f:
#     json.dump(samples, f, indent=2, ensure_ascii=False)
# print(f"Saved {len(samples)} examples to {out_path}")

### Math: GSM8k

In [41]:
train_data

Dataset({
    features: ['text'],
    num_rows: 5153
})